# Warum ist Parakeet in Colab langsam?

Separates Diagnose-Notebook für **Parakeet-TDT-0.6B-v3**, onnx-asr 0.12.0 und ORT GPU 1.27.0. Es verändert keine normalen Transkriptionsvorgaben oder vorhandenen Folgen.

1. In Colab einen **GPU-Laufzeittyp** wählen und den aktuellen V3-Checkout unter `/content/podcast_words_v3` bereitstellen. Dieser Checkout muss `src/podcast_words/asr_diagnostics.py` enthalten; ein altes Notebook allein genügt nicht.
2. Unten bei Bedarf eine bereits vorhandene Originaldatei angeben. Sonst die UFO537-Quelle aus dem lokalen Katalog einmal herunterladen. Die neue Messreihe benötigt keinen historischen CPU-Hash.
3. Zellen der Reihe nach ausführen. Standard: **120 Sekunden ab Minute 5**, eine vollständige Aufwärmrunde und drei Messungen je Variante. Modellgewichte werden außerhalb der Erkennungszeit geladen; FP32 benötigt zusätzlich etwa 2,5 GB Download-/Cacheplatz.
4. Am Ende **`diagnostic-report.json`** oder das kleine **`diagnostic-results.zip`** zurückgeben. Beide werden nach jedem Versuch aktualisiert und optional in einem eigenen Drive-Unterordner gesichert. Audio, Gewichte und Rohprofile bleiben auf der VM.

Hypothesen: (1) Der dynamische INT8-Export verschiebt teure Berechnungen auf die CPU. (2) Viele kleine Aufrufe und Kopien bremsen zusätzlich. (3) Vier Threads passen schlecht zum knappen Colab-Host. Erst messen, dann jeweils eine Einstellung ändern. Kein vollständiger Folgenlauf und kein automatischer Modellwechsel.

FP32-Ladekorrektur vom 24.10.2026: Der Messhelfer fordert für FP32 normale lokale Dateien an. Dadurch liegen `encoder-model.onnx` und seine externen Gewichte gemeinsam außerhalb des verlinkten Hub-Snapshots. Das behebt den reproduzierten Fehler `External data path escapes model directory`, ohne die ONNX-Pfadprüfung abzuschalten. Die Vorbereitung kann bereits gecachte Bytes kopieren; sie liegt außerhalb der Inferenzmessung und benötigt zusätzlichen lokalen Speicher.

Für den nächsten Versuch eine **neue Messreihe** mit diesem Notebook und dem aktualisierten Messhelfer starten. Als `source_audio_file` kann die zuvor eingefrorene `source.audio` dienen, solange die alte VM noch verfügbar ist. Neue und alte Berichte mit unterschiedlichen Helfer-Codehashes werden nicht stillschweigend verglichen. Die Standardvarianten erneut messen; die fehlende FP32-Messung ist noch kein Beleg für oder gegen dessen Geschwindigkeit.


In [ ]:
import json
import shutil
import subprocess
import sys
import time
import uuid
import zipfile
from pathlib import Path

project = Path("/content/podcast_words_v3")
source_audio_file = None  # Optional: "/content/drive/MyDrive/.../source-audio.source"
episode_id = "ep-5764a1149f18a8f68698fb45bb405264"
excerpt_start_seconds = 300
excerpt_duration_seconds = 120
profile_duration_seconds = 8  # Höchstens 15; bei Stille einen anderen Ausschnitt wählen.
repeats = 3

# Standardversuch. Reihenfolge für eine Gegenprobe in einer neuen Messreihe umkehren.
timing_cases = ["cuda-int8", "cuda-fp32", "cpu-int8"]
profile_cases = ["cuda-int8", "cuda-fp32"]

# Optional erst nach dem Standardvergleich aktivieren; jede Variante hat eine Bezugsvariante.
extra_cases = []
# extra_cases = ["cuda-int8-threads2", "cuda-fp32-vad-cpu",
#                "cuda-fp32-batch4", "cuda-fp32-preprocess-cpu"]

save_to_drive = True
drive_parent = Path("/content/drive/MyDrive/podcast-words-v3/diagnostics")
experiment_id = uuid.uuid4().hex
work_parent = Path("/content/podcast-words-diagnostics")
work_dir = work_parent / experiment_id
log_dir = work_parent / "logs" / experiment_id
log_dir.mkdir(parents=True, exist_ok=True)
if not (project / "src/podcast_words/asr_diagnostics.py").is_file():
    raise FileNotFoundError("Aktuellen V3-Checkout inklusive asr_diagnostics.py bereitstellen.")
if not 3 <= repeats <= 7:
    raise ValueError("Drei bis sieben Wiederholungen verwenden.")
print("Messreihe:", experiment_id)

## Umgebung einrichten und GPU prüfen

Der Notebook-Kernel und das ASR-Python sind getrennte Interpreter. Die Berechnungen laufen über Python 3.12 aus der mit `uv.lock` eingerichteten GPU-Umgebung. Fehlende GPU-/CUDA-Bibliotheken dürfen nicht unbemerkt zum CPU-Vergleich werden. Die tatsächliche Knotenausführung wird später separat profiliert.


In [ ]:
import re

if not shutil.which("ffmpeg"):
    raise RuntimeError("FFmpeg fehlt in dieser Colab-VM.")
gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=True)
driver = re.search(r"Driver Version:\s*(\d+)", gpu.stdout)
if not driver or int(driver.group(1)) < 580:
    raise RuntimeError("Der gepinnte ORT-1.27-GPU-Lauf benötigt einen passenden Treiber ab 580.")
print(gpu.stdout)
if save_to_drive:
    from google.colab import drive

    drive.mount("/content/drive")
    drive_destination = drive_parent / experiment_id
    drive_destination.mkdir(parents=True, exist_ok=True)

setup_started = time.perf_counter()
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.10.9"], check=True)
subprocess.run(
    [
        sys.executable,
        "-m",
        "uv",
        "sync",
        "--locked",
        "--no-default-groups",
        "--group",
        "gpu",
        "--python",
        "3.12",
    ],
    cwd=project,
    check=True,
)
setup_seconds = time.perf_counter() - setup_started
asr_python = project / ".venv/bin/python"
command_prefix = [str(asr_python), "-m", "podcast_words.asr_diagnostics"]


def git_output(*args):
    result = subprocess.run(["git", *args], cwd=project, capture_output=True, text=True, check=True)
    return result.stdout.strip()


context = {
    "experiment_id": experiment_id,
    "repository_commit": git_output("rev-parse", "HEAD"),
    "repository_branch": git_output("branch", "--show-current"),
    "repository_status": git_output("status", "--short"),
    "dependency_setup_seconds": setup_seconds,
    "notebook_python": sys.version,
    "timing_cases": timing_cases,
    "profile_cases": profile_cases,
    "extra_cases": extra_cases,
    "repeats": repeats,
    "external_failures": [],
}
preflight = subprocess.run(
    [
        str(asr_python),
        "-c",
        "import json; from podcast_words.asr_diagnostics import runtime_facts; print(json.dumps(runtime_facts()))",
    ],
    cwd=project,
    capture_output=True,
    text=True,
    check=True,
)
facts = json.loads(preflight.stdout)
if "CUDAExecutionProvider" not in facts["available_providers"]:
    raise RuntimeError("CUDAExecutionProvider fehlt. Keine Messung gestartet.")
print(json.dumps(facts, indent=2))
print("Setup außerhalb der Messungen:", round(setup_seconds, 2), "Sekunden")

## Audio einmal vorbereiten

Die Originaldatei wird eingefroren und gehasht. FFmpeg erzeugt einmal mono PCM16/16 kHz; alle Varianten verwenden anschließend dieselbe WAV-Datei. Der kurze Profilausschnitt enthält die identischen ersten PCM-Samples. Vorbereitung und Download zählen nicht zur Erkennungszeit.

Bei `source_audio_file=None` stammt der Link aus dem bestehenden Katalog. Eine bereits vorhandene Originaldatei ist vorzuziehen, wenn eine frühere Audiofassung verglichen werden soll. Den Ausschnitt kurz anhören: Er muss tatsächlich Sprache enthalten.


In [ ]:
from IPython.display import Audio, Markdown, display


def run_command(arguments, label, timeout=1200):
    log_path = log_dir / f"{label}.log"
    print(f"Starte {label} …", flush=True)
    started = time.perf_counter()
    try:
        with log_path.open("w", encoding="utf-8") as stream:
            result = subprocess.run(
                [*command_prefix, *arguments],
                cwd=project,
                stdout=stream,
                stderr=subprocess.STDOUT,
                text=True,
                timeout=timeout,
                check=False,
            )
        ok = result.returncode == 0
        if not ok:
            context["external_failures"].append({"label": label, "returncode": result.returncode})
    except subprocess.TimeoutExpired:
        ok = False
        context["external_failures"].append(
            {"label": label, "error": "timeout", "seconds": timeout}
        )
    print(log_path.read_text(encoding="utf-8", errors="replace")[-5000:])
    print(f"{label}: {'fertig' if ok else 'FEHLER'} nach {time.perf_counter() - started:.1f}s")
    return ok


prepare_args = [
    "prepare",
    "--work-dir",
    str(work_dir),
    "--start",
    str(excerpt_start_seconds),
    "--duration",
    str(excerpt_duration_seconds),
    "--profile-duration",
    str(profile_duration_seconds),
]
if source_audio_file is not None:
    prepare_args += ["--audio-file", str(source_audio_file)]
else:
    catalog = json.loads((project / "data/podcast-ufo/catalog.json").read_bytes())
    episode = next(item for item in catalog["episodes"] if item["episode_id"] == episode_id)
    prepare_args += ["--audio-url", episode["enclosures"][0]["url"]]
if not run_command(prepare_args, "prepare"):
    raise RuntimeError("Audiovorbereitung fehlgeschlagen; Details stehen oben und im Log.")
manifest = json.loads((work_dir / "manifest.json").read_bytes())
print(json.dumps(manifest, indent=2, ensure_ascii=False))

display(Audio(filename=str(work_dir / "profile.wav")))


def checkpoint():
    context_path = work_dir / "notebook-context.json"
    context_path.write_text(json.dumps(context, indent=2) + "\n", encoding="utf-8")
    if not run_command(["summarize", "--work-dir", str(work_dir)], "summary", timeout=120):
        raise RuntimeError("Bericht konnte nicht gültig zusammengeführt werden; Logs prüfen.")
    bundle = work_dir / "diagnostic-results.zip"
    with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for filename in ("diagnostic-report.json", "manifest.json", "notebook-context.json"):
            archive.write(work_dir / filename, filename)
        for path in sorted(log_dir.glob("*.log")):
            archive.write(path, f"logs/{path.name}")
    if save_to_drive:
        for path in (work_dir / "diagnostic-report.json", bundle):
            temporary = drive_destination / f"{path.name}.partial"
            shutil.copy2(path, temporary)
            temporary.replace(drive_destination / path.name)
        print("Bericht und ZIP gesichert:", drive_destination)


checkpoint()

## Standardvergleich: INT8, FP32 und CPU-Kontrolle

Jede Variante läuft in einem eigenen Prozess, lädt ihr Modell einmal, transkribiert den gesamten Ausschnitt einmal zum Vorwärmen und danach dreimal zur Messung. Hashprüfung, Validierung, Modellaufbau und Drive-Kopien liegen außerhalb der Erkennungszeit. Alle Einzelwerte, Median, Spannweite, CPU-Prozesszeit und Text-/Zeitstabilität werden gespeichert. Eine Variante überspringt keine Inferenz wegen eines vorhandenen Transkripts.

Die CPU-Kontrolle läuft auf derselben VM. onnx-asr verwendet dort standardmäßig NumPy-Vorverarbeitung und auf CUDA ONNX-Conv-Vorverarbeitung; dieser Vergleich beschreibt daher die beiden Laufwege und isoliert nicht allein den Provider. Ein fehlgeschlagener Versuch wird gespeichert; die übrigen Varianten können weiterlaufen. Ein Timeout kann einen als `running` gespeicherten Teilbericht hinterlassen und wird zusätzlich im Notebookkontext vermerkt.


In [ ]:
for case_name in timing_cases:
    run_command(
        ["run", "--work-dir", str(work_dir), "--case", case_name, "--repeats", str(repeats)],
        case_name,
    )
    checkpoint()

## Kurze Profile: Wo wird tatsächlich gerechnet?

Je Variante wird **nur der kurze Präfix** einmal mit Profilierung verarbeitet. Das ist ein Nachweis der ausgeführten Operationen, keine zweite Langzeitmessung. Es gibt dabei keine Aufwärmrunde und keinen Vergleich dieser Laufzeit mit dem längeren unprofilierten Ausschnitt. Encoder, Decoder, VAD und vorhandene Vorverarbeitungs-/Resampler-Sessions werden separat zusammengefasst.

Der Bericht kennzeichnet Profile am ORT-Ereignislimit als unvollständig. Fehlende `Kernel`-Ereignisse bedeuten ungemessene Gerätezeit; Host-Dauern und Knotenzahlen ergeben keine CPU/GPU-Zeitprozente. Rohprofile werden nicht nach Drive kopiert.


In [ ]:
for case_name in profile_cases:
    run_command(
        ["run", "--work-dir", str(work_dir), "--case", case_name, "--profile"],
        f"{case_name}-profile",
    )
    checkpoint()

## Weitere Einzeltests (optional)

`extra_cases` oben zunächst leer lassen. Nach dem ersten Vergleich gezielt aktivieren:

| Variante | Vergleich mit | Geänderte Einstellung |
| --- | --- | --- |
| `cuda-int8-threads2` | `cuda-int8` | zwei statt vier Intra-Op-Threads |
| `cuda-fp32-vad-cpu` | `cuda-fp32` | VAD auf CPU |
| `cuda-fp32-batch4` | `cuda-fp32` | vier statt einem Sprachsegment je Batch |
| `cuda-fp32-preprocess-cpu` | `cuda-fp32` | NumPy-Vorverarbeitung auf CPU |

Für eine ausgewählte Zusatzvariante kann unten `profile=True` nicht einfach bei den Zeitmessungen zugeschaltet werden; stattdessen ihren Namen in `profile_cases` aufnehmen und die Profilzelle gesondert ausführen. Ein größerer Batch kann mehr GPU-Speicher benötigen. Fehler und unbestätigte GPU-Ausführung sind Ergebnisse, kein Geschwindigkeitsnachweis.


In [ ]:
for case_name in extra_cases:
    run_command(
        ["run", "--work-dir", str(work_dir), "--case", case_name, "--repeats", str(repeats)],
        case_name,
    )
    checkpoint()

## Ergebnisse ansehen und zurückgeben

Erst eine klare Verbesserung bei gültigem Text und gültigen Zeitmarken auf längeres Audio übertragen. Textähnlichkeit ist ohne abgehörte Referenz keine Genauigkeitsmessung. Die Reihenfolge und eine geteilte Colab-VM können Messwerte beeinflussen; einen interessanten Vergleich in einer neuen Messreihe mit umgekehrter Reihenfolge wiederholen.

Nach jedem Versuch liegt bereits eine Zwischenfassung auf Drive. Die abschließende JSON enthält auch Fehlversuche und kurze Profilzusammenfassungen; das ZIP ergänzt Logs. Bitte eine dieser beiden Dateien zur Auswertung zurückgeben.


In [ ]:
checkpoint()
report_path = work_dir / "diagnostic-report.json"
report = json.loads(report_path.read_bytes())
rows = ["| Variante | n | Median (s) | Min-Max (s) | RTF |", "| --- | ---: | ---: | ---: | ---: |"]
for result in report["reports"]:
    if result["status"] != "completed":
        print("Unvollständig/Fehler:", result["case_name"], result.get("error", result["status"]))
        continue
    stats = result.get("statistics")
    if stats:
        rows.append(
            f"| {result['case_name']} | {stats['n']} | {stats['median_seconds']:.3f} | {stats['min_seconds']:.3f}-{stats['max_seconds']:.3f} | {stats['median_real_time_factor']:.3f} |"
        )
        print(result["case_name"], "Textprobe:", result["sample_output"]["text"][:400])
    if result.get("profile"):
        proof = result["profile"]
        print(
            result["case_name"],
            "CUDA-Modellknoten:",
            proof["cuda_model_nodes"],
            "Ereignislimit erreicht:",
            proof["event_limit_reached"],
        )
        if not proof["cuda_execution_confirmed"]:
            print(
                "ACHTUNG: Für diesen kurzen Profilausschnitt ist keine CUDA-Modellausführung belegt."
            )
display(Markdown("\n".join(rows)))
print(json.dumps(report["controlled_comparisons"], indent=2, ensure_ascii=False))
print("JSON:", report_path)
print("ZIP:", work_dir / "diagnostic-results.zip")
if save_to_drive:
    print("Drive:", drive_destination)
# Optional ohne Drive direkt herunterladen:
# from google.colab import files
# files.download(str(work_dir / "diagnostic-results.zip"))